## 1. Extract Protein and CDS Sequences from GenBank Files

In [ ]:
from Bio import SeqIO
import os

input_dir = '/public/home/luoyt/Data/Ral/allral_genome/1267/1267/file/'
output_dir = '/public/home/luoyt/Data/Ral/allral_genome/1267/1267/protein_and_cds/'
    
for file in os.listdir(input_dir):
    if not file.endswith('.gbff'):
        continue
    
    name = file.replace('.gbff', '')
    print(f"Processing {file}...")
    
    proteins = []
    cds = []
    
    for record in SeqIO.parse(os.path.join(input_dir, file), "genbank"):
        for feature in record.features:
            if feature.type != "CDS":
                continue
            
            # 跳过假基因和不完整的
            if "pseudo" in feature.qualifiers:
                continue
            if "<" in str(feature.location) or ">" in str(feature.location):
                continue
            
            gene_id = feature.qualifiers.get("locus_tag", ["unknown"])[0]
            
            # 提取蛋白
            if "translation" in feature.qualifiers:
                protein = feature.qualifiers["translation"][0]
                proteins.append(f">{gene_id}\n{protein}")
            
            # 提取 CDS（从基因组序列里截取）
            try:
                cds_seq = str(feature.location.extract(record.seq))
                cds.append(f">{gene_id}\n{cds_seq}")
            except Exception as e:
                print(f"  Warning: Failed to extract CDS for {gene_id}: {e}")
    
    # 写入文件
    with open(os.path.join(output_dir, f"{name}.faa"), "w") as f:
        f.write("\n".join(proteins))
    with open(os.path.join(output_dir, f"{name}.fna"), "w") as f:
        f.write("\n".join(cds))
    
    print(f"  ✓ {name}.faa ({len(proteins)} proteins)")
    print(f"  ✓ {name}.fna ({len(cds)} CDS)")

## 2.Run BUSCO

In [ ]:
#!/bin/bash
#PBS -N busco_batch1
#PBS -l nodes=node13:ppn=2,walltime=1000:00:00
#PBS -j oe
#PBS -o /public/home/luyt/Data/Ral/allral_genome/1265_busco_tree/log/busco.log
#PBS -q low
#PBS -t 501-1121%40

cd $PBS_O_WORKDIR

source /public/home/luyt/miniconda3/bin/activate BUSCO

echo "Job started at: $(date '+%Y-%m-%d %H:%M:%S')"

# 获取文件列表
genome_list="/public/home/luyt/Data/Ral/allral_genome/1265_busco_tree/genome_id.list"

# 根据任务ID取文件列表的第几行
file_name=$(sed -n "$((PBS_ARRAYID))p" "$genome_list")
file="/public/home/luyt/Data/Ral/allral_genome/1267/1267/protein_and_cds/${file_name}.faa"

echo "Processing ${file_name}"

busco -i "$file" \
      -l /public/home/luyt/Data/Ral/busco_db/ralstonia_odb12 \
      -m proteins \
      -o result/${file_name} \
      -c 2 \
      -f \
      --offline \
      -q

## 3.check busco results

In [ ]:
#!/usr/bin/env python3
import os
import re

busco_result_dir = "/public/home/luyt/Data/Ral/allral_genome/1265_busco_tree/result"
output_file = "busco_quality_report.tsv"

print(f"Scanning BUSCO results in {busco_result_dir}...")

results = []

for busco_dir in sorted(os.listdir(busco_result_dir)):
    busco_path = os.path.join(busco_result_dir, busco_dir)
    
    if not os.path.isdir(busco_path):
        continue
    
    # 找 short_summary 文件
    summary_file = None
    for file in os.listdir(busco_path):
        if file.startswith("short_summary") and file.endswith(".txt"):
            summary_file = os.path.join(busco_path, file)
            break
    
    if not summary_file:
        print(f"⚠️  {busco_dir}: No summary file found")
        continue
    
    with open(summary_file, 'r') as f:
        content = f.read()
    
    # 提取所有值
    c_match = re.search(r'C:(\d+\.?\d*)%', content)
    s_match = re.search(r'S:(\d+\.?\d*)%', content)
    d_match = re.search(r'D:(\d+\.?\d*)%', content)
    f_match = re.search(r'F:(\d+\.?\d*)%', content)
    m_match = re.search(r'M:(\d+\.?\d*)%', content)
    
    if not c_match:
        continue
    
    completeness = float(c_match.group(1))
    single_copy = float(s_match.group(1)) if s_match else 0
    duplicated = float(d_match.group(1)) if d_match else 0
    fragmented = float(f_match.group(1)) if f_match else 0
    missing = float(m_match.group(1)) if m_match else 0
    
    results.append({
        'genome': busco_dir,
        'complete': completeness,
        'single_copy': single_copy,
        'duplicated': duplicated,
        'fragmented': fragmented,
        'missing': missing
    })
    
    print(f"✓ {busco_dir}: C={completeness}%, S={single_copy}%, M={missing}%")

# 写 TSV
with open(output_file, 'w') as f:
    f.write("Genome\tComplete(%)\tSingle-Copy(%)\tDuplicated(%)\tFragmented(%)\tMissing(%)\n")
    for r in sorted(results, key=lambda x: x['complete'], reverse=True):
        f.write(f"{r['genome']}\t{r['complete']:.1f}\t{r['single_copy']:.1f}\t{r['duplicated']:.1f}\t{r['fragmented']:.1f}\t{r['missing']:.1f}\n")

print(f"\n✅ Report saved to {output_file}")
print(f"Total genomes: {len(results)}")

## 4. Identify Conserved Single-Copy BUSCO Genes

In [ ]:
for dir in result/GCF*/run_ralstonia_odb12/busco_sequences/single_copy_busco_sequences/; do
    ls "$dir" | sed 's/.faa//'
done | sort | uniq -c | sort -rn > busco_counts2.txt

awk '$1==1082 {print $2}' busco_counts2.txt > conserved_buscos.txt

## 6. Concatenate BUSCO Protein Sequences by Gene

In [ ]:
#!/usr/bin/env python3
import os
import glob
import re

output_file = "busco_orthologs_matrix.tsv"

# 读基因列表
with open("conserved_buscos.txt") as f:
    genes = [line.strip() for line in f]

# 读所有菌株的数据
strains = []
data = {}

for file in sorted(glob.glob("final_result/GCF*/run_ralstonia_odb12/full_table.tsv")):
    strain = os.path.basename(os.path.dirname(os.path.dirname(file)))
    strains.append(strain)
    print(f"Loading {strain}...")
    
    data[strain] = {}
    with open(file) as f:
        for i, line in enumerate(f):
            if line.startswith("#"):
                continue
            
            line = line.strip()
            if not line:  # 跳过空行
                continue
            
            # 用 \s+ 分割（任意空格）
            parts = re.split(r'\s+', line)
            
            if len(parts) < 3:
                continue
            
            gene = parts[0]
            locus = parts[2]
            data[strain][gene] = locus

# 写矩阵
with open(output_file, "w") as out:
    out.write("Gene\t" + "\t".join(strains) + "\n")
    
    for gene in genes:
        row = [gene]
        for strain in strains:
            locus = data[strain].get(gene, "NA")
            row.append(locus)
        out.write("\t".join(row) + "\n")

print(f"Done! Output: {output_file}")

In [ ]:
#!/usr/bin/env python3
"""
根据BUSCO基因表，从各GCF的CDS fna文件中提取序列，
每个BUSCO基因的所有样本序列合并到一个fasta文件中。
缺失序列记录到log文件。
"""

import os
import sys
from collections import defaultdict

# ========== 配置路径，按需修改 ==========
TSV_FILE    = "/public/home/luoyt/Data/Ral/allral_tree/busco/busco_orthologs_matrix.tsv"           # TSV表文件路径
CDS_DIR     = "/public/home/luoyt/Data/Ral/allral_genome/1267/1267/protein_and_cds"
OUTPUT_DIR  = "/public4/luoyt/ral/1082_tree/sequence/raw"                # 输出目录
LOG_FILE    = "/public4/luoyt/ral/1082_tree/sequence/extract_log.txt"
# ========================================

# ========================================
 
def parse_fna(fna_path):
    """解析fasta文件，返回 ({gene_id: sequence}, [空序列的gene_id列表])"""
    seqs = {}
    empty_ids = []
    current_id = None
    current_seq = []
    with open(fna_path) as f:
        for line in f:
            line = line.rstrip()
            if line.startswith(">"):
                if current_id is not None:
                    seq = "".join(current_seq)
                    if seq:
                        seqs[current_id] = seq
                    else:
                        empty_ids.append(current_id)  # 有header但序列为空
                current_id = line[1:].split()[0]
                current_seq = []
            else:
                current_seq.append(line)
    # 处理最后一条
    if current_id is not None:
        seq = "".join(current_seq)
        if seq:
            seqs[current_id] = seq
        else:
            empty_ids.append(current_id)
    return seqs, empty_ids
 
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
 
    # 读取TSV表
    print(f"读取TSV表: {TSV_FILE}")
    with open(TSV_FILE) as f:
        lines = f.read().splitlines()
 
    header = lines[0].split("\t")
    gcf_list = header[1:]  # 第一列是Gene，后面都是GCF号
    total_samples = len(gcf_list)
    print(f"共 {total_samples} 个GCF样本，{len(lines)-1} 个BUSCO基因")
 
    # 预加载所有GCF的fna序列（按需加载，避免内存爆炸）
    # 先建立 GCF -> fna路径 的映射
    gcf_to_fna = {}
    for gcf in gcf_list:
        fna_path = os.path.join(CDS_DIR, f"{gcf}.fna")
        if os.path.exists(fna_path):
            gcf_to_fna[gcf] = fna_path
        else:
            gcf_to_fna[gcf] = None
 
    missing_fna = [g for g, p in gcf_to_fna.items() if p is None]
    if missing_fna:
        print(f"警告：以下 {len(missing_fna)} 个GCF没有对应的fna文件：")
        for g in missing_fna:
            print(f"  {g}")
 
    # 缓存已加载的fna（全部预加载）
    print("加载所有fna文件...")
    gcf_seqs = {}
    gcf_empty_ids = {}  # 记录各GCF中序列为空的基因ID
    for gcf, fna_path in gcf_to_fna.items():
        if fna_path:
            seqs, empty_ids = parse_fna(fna_path)
            gcf_seqs[gcf] = seqs
            if empty_ids:
                gcf_empty_ids[gcf] = set(empty_ids)
        else:
            gcf_seqs[gcf] = {}
    print("加载完成，开始提取序列...")
 
    # 把空序列情况写入log开头
    log_lines = []
    if gcf_empty_ids:
        log_lines.append("=" * 60)
        log_lines.append("[WARN] 以下GCF文件中存在header有序列名但序列为空的条目：")
        for gcf, ids in sorted(gcf_empty_ids.items()):
            log_lines.append(f"  {gcf}: {', '.join(sorted(ids))}")
        log_lines.append("=" * 60)
        log_lines.append("")
 
    # 遍历每个BUSCO基因（每一行）
    for row_line in lines[1:]:
        cols = row_line.split("\t")
        busco_gene = cols[0]
        gene_ids   = cols[1:]  # 与gcf_list一一对应
 
        out_path = os.path.join(OUTPUT_DIR, f"{busco_gene}.fna")
        found_count = 0
        missing_samples = []
 
        with open(out_path, "w") as out_f:
            for gcf, gene_id in zip(gcf_list, gene_ids):
                gene_id = gene_id.strip()
 
                # 跳过空值
                if not gene_id or gene_id in ("", "-", "N/A"):
                    missing_samples.append(f"{gcf}(空值)")
                    continue
 
                seq = gcf_seqs[gcf].get(gene_id)
                if seq:
                    out_f.write(f">{gcf}|{gene_id}\n{seq}\n")
                    found_count += 1
                elif gene_id in gcf_empty_ids.get(gcf, set()):
                    missing_samples.append(f"{gcf}({gene_id})[序列为空]")
                else:
                    missing_samples.append(f"{gcf}({gene_id})[ID不存在]")
 
        if found_count < total_samples:
            missing_str = ", ".join(missing_samples)
            log_lines.append(
                f"[MISSING] {busco_gene}: 找到 {found_count}/{total_samples} 条序列。"
                f" 缺失样本: {missing_str}"
            )
        else:
            log_lines.append(f"[OK] {busco_gene}: {found_count}/{total_samples} 条序列全部找到")
 
    # 写log
    with open(LOG_FILE, "w") as lf:
        lf.write("\n".join(log_lines) + "\n")
 
    total_missing = sum(1 for l in log_lines if l.startswith("[MISSING]"))
    total_ok      = sum(1 for l in log_lines if l.startswith("[OK]"))
    print(f"\n完成！输出目录: {OUTPUT_DIR}")
    print(f"  全部齐全: {total_ok} 个基因")
    print(f"  有缺失:   {total_missing} 个基因")
    print(f"  日志文件: {LOG_FILE}")
 
if __name__ == "__main__":
    main()

In [ ]:
#!/bin/bash
#PBS -N mafft_trimal1
#PBS -l nodes=node5:ppn=5,walltime=1000:00:00
#PBS -j oe
#PBS -o /public4/luoyt/ral/1082_tree/sequence/mafft_trim/log/mafft_trimal.log
#PBS -q normal
#PBS -t 131-277%10

cd $PBS_O_WORKDIR

source /public/home/luoyt/miniconda3/bin/activate tree

echo "Job started at: $(date '+%Y-%m-%d %H:%M:%S')"
echo "ARRAYID=$PBS_ARRAYID"

list="/public4/luoyt/ral/1082_tree/sequence/list.txt"
filepath1="/public4/luoyt/ral/1082_tree/sequence/raw"
filepath2="/public4/luoyt/ral/1082_tree/sequence/mafft_trim/result"
# 根据任务ID取文件列表的第几行
file_name=$(sed -n "$((PBS_ARRAYID))p" "$list")
name=${file_name%.fna}

echo "=== Task $PBS_ARRAYID: 处理 ${name} ==="

mafft --globalpair --maxiterate 1000 --thread 5 \
    ${filepath1}/${file_name} > ${filepath2}/${name}_mafft.faa

trimal -in ${filepath2}/${name}_mafft.faa \
       -out ${filepath2}/${name}_trim.faa \
       -gappyout

echo "Job finished at: $(date '+%Y-%m-%d %H:%M:%S')"

In [ ]:
/public/home/luoyt/miniconda3/envs/tree/bin/AMAS.py concat \
    -i /public4/luoyt/ral/1082_tree/sequence/mafft_trim/result/*_trim.faa \
    -f fasta \
    -d dna \
    -t /public4/luoyt/ral/1082_tree/sequence/mafft_trim/supermatrix.faa \
    -p /public4/luoyt/ral/1082_tree/sequence/mafft_trim/partitions.txt \
    --part-format nexus

In [ ]:
#!/bin/bash
#PBS -N iqtree
#PBS -l nodes=node21:ppn=64,walltime=1000:00:00
#PBS -j oe
#PBS -o /public4/luoyt/ral/1082_tree/iqtree/iqtree.log
#PBS -q normal

cd $PBS_O_WORKDIR

source /public/home/luoyt/miniconda3/bin/activate tree

echo "Job started at: $(date '+%Y-%m-%d %H:%M:%S')"

faa="/public4/luoyt/ral/1082_tree/sequence/mafft_trim/supermatrix.faa"
partions="/public4/luoyt/ral/1082_tree/sequence/mafft_trim/partitions.txt"

iqtree3 -s ${faa} \
        -p ${partions} \
        -m MFP \
        -B 1000 -alrt 1000 \
        -T 64 \
        --prefix allral_1082 \
        -safe

echo "Job finished at: $(date '+%Y-%m-%d %H:%M:%S')"
